# Where The Input Work Lives

## DataLoader and DDP lessons for support staff

A live Binder/Jupyter teaching deck about how bytes become training throughput.

Core opinion: start boring, measure where the toll is, move work only when the evidence says it matters, and validate the candidate in DDP before recommending a training path.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.options.display.max_colwidth = 120
pio.renderers.default = "notebook_connected"


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data/generated").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    return Path.cwd().resolve()

ROOT = find_repo_root()
DATA = ROOT / "data/generated"

COLORS = {
    "cpu": "#2f6f73",
    "dali": "#b45f06",
    "uint8": "#4c78a8",
    "fp16": "#f58518",
    "synthetic": "#7a5195",
    "line": "#5f6368",
    "storage": "#6b7280",
}


def csv(path):
    return pd.read_csv(DATA / path)


def num(value):
    if pd.isna(value):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).strip().replace(",", "").replace("%", "").replace("x", "")
    return float(text)


def with_numbers(df, columns):
    out = df.copy()
    for column in columns:
        out[column] = out[column].map(num)
    return out


def clean_layout(fig, title, height=520):
    fig.update_layout(
        title=title,
        height=height,
        template="plotly_white",
        margin=dict(l=55, r=35, t=80, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        font=dict(size=15),
    )
    return fig


def pipeline_sankey():
    nodes = [
        "Storage", "CPU memory / page cache", "CPU workers", "CPU batch",
        "H2D copy", "GPU memory", "Benchmark loop", "DALI CPU reader",
        "DALI mixed decode", "DALI GPU ops", "Offline prep already paid",
        "NumPy tensor read", "Synthetic GPU generator",
    ]
    node_color = [
        "#9ca3af", "#cbd5e1", "#2f6f73", "#93c5fd", "#64748b", "#f59e0b", "#111827",
        "#b45f06", "#d97706", "#f59e0b", "#8b5cf6", "#4c78a8", "#7a5195",
    ]
    flows = {
        "PyTorch CPU JPEG": ([0, 1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6], [3, 3, 3, 3, 3, 3]),
        "DALI JPEG": ([0, 7, 8, 9, 5], [7, 8, 9, 5, 6], [3, 3, 3, 3, 3]),
        "NumPy uint8 prepared": ([10, 0, 11, 2, 3, 4, 5], [0, 11, 2, 3, 4, 5, 6], [2, 2, 2, 2, 2, 2, 2]),
        "NumPy fp16 ceiling": ([10, 0, 11, 3, 4, 5], [0, 11, 3, 4, 5, 6], [2, 2, 2, 2, 2, 2]),
        "Synthetic GPU ceiling": ([12, 5], [5, 6], [3, 3]),
    }
    fig = go.Figure()
    for index, (name, (source, target, value)) in enumerate(flows.items()):
        fig.add_trace(go.Sankey(
            visible=index == 0,
            arrangement="snap",
            node=dict(label=nodes, pad=16, thickness=18, color=node_color),
            link=dict(source=source, target=target, value=value, color="rgba(47,111,115,0.28)"),
            name=name,
        ))
    buttons = []
    for index, name in enumerate(flows):
        visible = [False] * len(flows)
        visible[index] = True
        buttons.append(dict(label=name, method="update", args=[{"visible": visible}, {"title": f"Where the input work lives: {name}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    return clean_layout(fig, "Where the input work lives: PyTorch CPU JPEG", height=560)


def worker_scan_fig():
    frames = []
    for platform, path in [
        ("B200", "dataloader/b200-single-node-replicated-table-2.csv"),
        ("RTX Pro 6000", "dataloader/rtx-single-node-replicated-table-2.csv"),
    ]:
        df = with_numbers(csv(path), ["samples_s", "retained_imbalance", "retained_jitter"])
        df["platform"] = platform
        frames.append(df)
    data = pd.concat(frames, ignore_index=True)
    combos = [(p, b) for p in data.platform.unique() for b in sorted(data.loc[data.platform == p, "batch"].unique())]
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for i, (platform, batch) in enumerate(combos):
        sub = data[(data.platform == platform) & (data.batch == batch)].sort_values("workers")
        visible = i == 0
        fig.add_trace(go.Scatter(x=sub.workers, y=sub.samples_s, mode="lines+markers", name="samples/s", visible=visible, marker_color=COLORS["cpu"]), secondary_y=False)
        fig.add_trace(go.Scatter(x=sub.workers, y=sub.retained_imbalance, mode="lines+markers", name="rank imbalance %", visible=visible, marker_color=COLORS["line"], line=dict(dash="dot")), secondary_y=True)
    buttons = []
    for i, (platform, batch) in enumerate(combos):
        visible = [False] * (2 * len(combos))
        visible[2*i] = True
        visible[2*i + 1] = True
        buttons.append(dict(label=f"{platform}, batch {batch}", method="update", args=[{"visible": visible}, {"title": f"Workers feed eight ranks: {platform}, batch {batch}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_xaxes(title_text="num_workers per rank")
    fig.update_yaxes(title_text="samples/s", secondary_y=False)
    fig.update_yaxes(title_text="retained rank imbalance %", secondary_y=True)
    return clean_layout(fig, "Workers feed eight ranks: B200, batch 512", height=540)


def backend_crossover_fig():
    df = with_numbers(csv("dataloader/optimized-backend-crossover-table-2.csv"), ["cpu_samples_s", "dali_samples_s", "dali_cpu"])
    platforms = list(df.platform.unique())
    fig = go.Figure()
    for i, platform in enumerate(platforms):
        sub = df[df.platform == platform]
        visible = i == 0
        hover_cpu = [f"{row.endpoint}<br>{row.best_cpu_config}<br>{row.cpu_samples_s:,.0f} samples/s" for row in sub.itertuples()]
        hover_dali = [f"{row.endpoint}<br>{row.best_dali_config}<br>{row.dali_samples_s:,.0f} samples/s<br>DALI/CPU {row.dali_cpu:.2f}x" for row in sub.itertuples()]
        fig.add_bar(x=sub.endpoint, y=sub.cpu_samples_s, name="PyTorch CPU", marker_color=COLORS["cpu"], visible=visible, hovertext=hover_cpu, hoverinfo="text")
        fig.add_bar(x=sub.endpoint, y=sub.dali_samples_s, name="DALI", marker_color=COLORS["dali"], visible=visible, hovertext=hover_dali, hoverinfo="text")
    buttons = []
    for i, platform in enumerate(platforms):
        visible = [False] * (2 * len(platforms))
        visible[2*i] = True
        visible[2*i + 1] = True
        buttons.append(dict(label=platform, method="update", args=[{"visible": visible}, {"title": f"Backend choice changes with representation: {platform}"}]))
    fig.update_layout(barmode="group", updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_yaxes(title_text="DataLoader samples/s")
    return clean_layout(fig, "Backend choice changes with representation: B200", height=540)


def prepared_economics_fig():
    throughput = with_numbers(csv("dataloader/prepared-input-ceilings-table-2.csv"), ["uint8_samples_s", "fp16_samples_s", "fp16_uint8"])
    storage = with_numbers(csv("dataloader/prepared-input-ceilings-table-3.csv"), ["uint8_dataset_gib", "fp16_dataset_gib"])
    platforms = list(throughput.platform.unique())
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for i, platform in enumerate(platforms):
        sub = throughput[throughput.platform == platform].sort_values("size")
        visible = i == 0
        fig.add_trace(go.Scatter(x=sub["size"], y=sub.uint8_samples_s, mode="lines+markers", name=f"{platform} uint8 samples/s", visible=visible, marker_color=COLORS["uint8"]), secondary_y=False)
        fig.add_trace(go.Scatter(x=sub["size"], y=sub.fp16_samples_s, mode="lines+markers", name=f"{platform} fp16 samples/s", visible=visible, marker_color=COLORS["fp16"]), secondary_y=False)
    fig.add_trace(go.Scatter(x=storage["size"], y=storage.uint8_dataset_gib, mode="lines+markers", name="uint8 GiB", marker_color=COLORS["storage"], line=dict(dash="dash"), visible=True), secondary_y=True)
    fig.add_trace(go.Scatter(x=storage["size"], y=storage.fp16_dataset_gib, mode="lines+markers", name="fp16 GiB", marker_color="#374151", line=dict(dash="dot"), visible=True), secondary_y=True)
    buttons = []
    trace_count = 2 * len(platforms) + 2
    for i, platform in enumerate(platforms):
        visible = [False] * trace_count
        visible[2*i] = True
        visible[2*i + 1] = True
        visible[-2] = True
        visible[-1] = True
        buttons.append(dict(label=platform, method="update", args=[{"visible": visible}, {"title": f"Prepared inputs: pay storage/precompute once, {platform}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_xaxes(title_text="prepared image/tensor size")
    fig.update_yaxes(title_text="DataLoader samples/s", secondary_y=False)
    fig.update_yaxes(title_text="derived dataset GiB", secondary_y=True)
    return clean_layout(fig, "Prepared inputs: pay storage/precompute once, B200", height=560)


def ddp_truth_fig():
    one = with_numbers(csv("ddp/dataloader-candidate-followup-table-2.csv"), ["pytorch_cpu_images_s", "dali_images_s", "dali_pytorch"])
    scale = with_numbers(csv("ddp/multinode-scale-validation-table-2.csv"), ["1_node_img_s", "2_node_img_s", "4_node_img_s"])
    platforms = list(one.platform.unique())
    fig = make_subplots(rows=1, cols=2, subplot_titles=("One-node candidate validation", "Multi-node scale check"))
    for i, platform in enumerate(platforms):
        visible = i == 0
        sub = one[one.platform == platform]
        fig.add_trace(go.Bar(x=sub.endpoint, y=sub.pytorch_cpu_images_s, name="PyTorch CPU images/s", marker_color=COLORS["cpu"], visible=visible), row=1, col=1)
        fig.add_trace(go.Bar(x=sub.endpoint, y=sub.dali_images_s, name="DALI images/s", marker_color=COLORS["dali"], visible=visible), row=1, col=1)
        ss = scale[scale.platform == platform]
        for _, row in ss.iterrows():
            fig.add_trace(go.Scatter(x=[1,2,4], y=[row["1_node_img_s"], row["2_node_img_s"], row["4_node_img_s"]], mode="lines+markers", name=row["candidate"], visible=visible), row=1, col=2)
    # Two bars plus two scale candidates per platform.
    traces_per_platform = 4
    buttons = []
    for i, platform in enumerate(platforms):
        visible = [False] * (traces_per_platform * len(platforms))
        for t in range(traces_per_platform):
            visible[i * traces_per_platform + t] = True
        buttons.append(dict(label=platform, method="update", args=[{"visible": visible}, {"title": f"DDP is the training truth: {platform}"}]))
    fig.update_layout(barmode="group", updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_xaxes(title_text="endpoint", row=1, col=1)
    fig.update_xaxes(title_text="nodes", row=1, col=2)
    fig.update_yaxes(title_text="images/s", row=1, col=1)
    fig.update_yaxes(title_text="images/s", row=1, col=2)
    return clean_layout(fig, "DDP is the training truth: B200", height=560)


def input_ceiling_fig():
    df = with_numbers(csv("ddp/input-ceilings-table-2.csv"), ["images_s"])
    platforms = list(df.platform.unique())
    sizes = sorted(df["size"].unique())
    combos = [(p, s) for p in platforms for s in sizes]
    fig = go.Figure()
    for i, (platform, size) in enumerate(combos):
        sub = df[(df.platform == platform) & (df["size"] == size)]
        fig.add_bar(x=sub.input, y=sub.images_s, name=f"{platform} {size}", visible=i == 0, marker_color=[COLORS["uint8"], COLORS["fp16"], COLORS["synthetic"]])
    buttons = []
    for i, (platform, size) in enumerate(combos):
        visible = [False] * len(combos)
        visible[i] = True
        buttons.append(dict(label=f"{platform}, {size}", method="update", args=[{"visible": visible}, {"title": f"Synthetic GPU input removes the real input path: {platform}, {size}"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=0, y=1.18, xanchor="left")])
    fig.update_yaxes(title_text="DDP images/s")
    return clean_layout(fig, "Synthetic GPU input removes the real input path: B200, 224", height=520)


## What Council Changed In This Pass

The R1 review was **POLISH / near-GO**, not a science blocker.

Accepted changes for this implementation:

- Use a small visual backbone instead of a long benchmark report.
- Make the live interactions explicit.
- Keep DDP to validation handoff.
- Keep the guardrails in the main story.


## The Core Question

When a training job is slow, where is the limiting work?

- Storage read.
- JPEG decode.
- CPU crop/resize/normalization.
- Worker queue depth and prefetch overlap.
- Host-to-device copy.
- DALI decode and GPU image operators.
- Model compute, backward pass, optimizer, and rank synchronization.

This deck teaches a way to answer that question without turning one benchmark row into a universal rule.


## The Toll Booth Model

Each sample pays a sequence of tolls. Tuning works by doing one of four things:

- **Parallelize** a toll with workers or DALI threads.
- **Overlap** a toll with prefetch or queue depth.
- **Move** a toll from CPU to GPU or from runtime to offline preparation.
- **Remove** a toll in a control path such as synthetic GPU input.

The live selector below changes the path, not the conclusion.


In [ ]:
pipeline_sankey().show()


## Metrics That Tell You Where To Look

- `samples/s`: end-to-end DataLoader loop throughput; the headline input-side metric.
- `load samples/s`: input work before the GPU transfer part of the measured path.
- `H2D samples/s`: host-to-device transfer throughput; if it is far above `samples/s`, H2D is probably not the bottleneck.
- `rank_imbalance_percent`: how uneven the ranks are; important once one GPU becomes eight ranks or multiple nodes.

The lesson: a larger `samples/s` row is useful, but it is not the whole diagnosis.


## Knobs And What They Move

| Knob | What it changes | Typical failure signal |
| --- | --- | --- |
| `batch_size` | Amortizes fixed overhead and changes memory pressure. | Bigger stops helping or rank balance worsens. |
| `num_workers` | Parallelizes CPU file read, decode, transform, and collation. | Too few starve GPUs; too many add overhead. |
| `prefetch_factor` | Keeps batches queued ahead of consumption. | Low queues expose input waits; high queues burn memory. |
| `pin_memory` | Helps CPU batch tensors move to GPU. | Matters only if H2D is in the limiting path. |
| `persistent_workers` | Avoids repeated worker startup cost. | Mostly matters across epoch boundaries or repeated loader construction. |
| DALI threads / queue depth | Controls DALI pipeline parallelism and overlap. | Can help large images; can also add latency or memory pressure. |


## Interactive Moment 1: Workers Feed Ranks

One GPU tuning is a surface. One node with eight ranks is where worker count starts to teach rank pressure.

Use the dropdown to compare platform and batch. The point is not the exact winner; the point is the shape:

- Workers raise throughput until the CPU side is sufficiently parallel.
- More workers after the plateau do not guarantee more throughput.
- Rank imbalance can move independently from the headline rate.


In [ ]:
worker_scan_fig().show()


## Representation Changes The Experiment

| Representation | What is on disk | What it teaches |
| --- | --- | --- |
| `canonical-224` | Original ImageNet-style JPEGs. | Ordinary online ImageNet input behavior. |
| `derived-jpeg-1024` | ImageNet-derived pre-resized JPEGs. | How larger JPEG decode/image pressure changes backend choice. |
| `decode-stress` | Synthetic large JPEGs. | Mechanism probe for compressed large-image decode pressure. |
| `prepared-input-ceiling` uint8 | NumPy arrays. | What remains after JPEG decode is moved offline. |
| `prepared-input-ceiling` fp16 | Normalized fp16 tensors. | What happens when decode and most normalization are paid offline. |
| `synthetic-gpu-ceiling` | Generated tensors already on GPU. | Training-side ceiling with the real input path removed. |

Do not compare these as if they were the same dataset strategy.


## PyTorch CPU JPEG Path

The boring baseline is still a serious system:

1. Read compressed JPEG bytes from storage into host-visible memory.
2. Decode, crop or resize, convert, normalize, and collate in CPU workers.
3. Stage a CPU batch and copy it to GPU memory.
4. Hand the batch to the benchmark or training loop.

For ordinary ImageNet-like input, this is the first path to tune because it is simple, reproducible, and strong in the public `canonical-224` evidence.


## DALI JPEG Path

DALI changes where decode and image operators run and how the pipeline overlaps work.

It does **not** make every JPEG workload faster. It also does **not** imply that normal ImageNet JPEG reads bypass CPU memory through GDS.

The usable question is narrower: is runtime JPEG decode and image processing large enough to amortize the DALI pipeline overhead for this input representation?


## Interactive Moment 2: The Backend Crossover

The public result is conditional:

- At ordinary `canonical-224`, tuned PyTorch CPU is the starting recommendation.
- At `derived-jpeg-1024`, tuned DALI becomes the strongest measured input-side candidate.

Use the dropdown to switch platform. The teaching point is that backend choice changed because the representation and image size changed.


In [ ]:
backend_crossover_fig().show()


## Canonical 224 Lesson

For ordinary `canonical-224` ImageNet on these platforms, start with tuned PyTorch CPU DataLoader.

In the standard ImageNet DALI study, DALI reached about `0.44x` of the B200 CPU anchor and `0.41x` of the RTX Pro 6000 CPU anchor in the DataLoader-only loop.

That is not an anti-DALI rule. It is evidence that DALI overhead was not amortized at this input shape.


## Large Derived JPEG Lesson

For `derived-jpeg-1024`, the work shifts.

The JPEG decode and image-processing pressure is large enough that DALI becomes the stronger input-side candidate: about `3.11x` CPU on B200 and `2.95x` CPU on RTX Pro 6000 in the tuned endpoint comparison.

This is supporting crossover evidence for a larger pre-resized JPEG representation, not a replacement for canonical ImageNet evidence.


## Tune Before You Judge

A fair backend comparison needs tuned endpoints, not one fixed configuration.

- CPU needs batch, worker, prefetch, pin-memory, and persistence choices.
- DALI needs batch, thread, and queue-depth choices.
- Large inputs change the cost model enough that a bad default can hide a real crossover.

The support-staff habit: tune each candidate enough to learn its shape, then compare the best defensible endpoints.


## Prepared Inputs: Pay Once Or Pay Every Run

Prepared inputs are an economics lesson.

- NumPy uint8 removes JPEG decode but keeps runtime conversion and normalization.
- NumPy fp16 moves decode and most normalization offline.
- The cost is storage, preparation time, fixed preprocessing semantics, and another artifact to manage.

This is useful when a workload repeats enough times that paying once can be cheaper than paying every training run.


## Interactive Moment 3: Prepared-Input Economics

The plot pairs throughput with storage footprint. Use the dropdown to switch platform.

The teaching point is not "use fp16 shards everywhere." The point is to see how much online work can be removed and how much storage/precompute debt that creates.


In [ ]:
prepared_economics_fig().show()


## The Narrow Prepared-Tensor GDS Boundary

DALI NumPy CPU/GPU reader rows, when present, are a prepared-tensor transport probe.

They are not evidence that normal DALI JPEG uses GDS, not evidence that JPEG decode improved through GDS, and not a general no-CPU-memory input recipe.

Do not publish a GDS performance claim without same-node `verify-gds` evidence and paired CPU-reader versus GPU/O_DIRECT-reader rows on the same prepared-tensor path.


## DataLoader Is Candidate Selection

DataLoader-only throughput selects candidates and explains mechanisms.

Training throughput adds model compute, backward pass, optimizer work, rank timing, distributed sharding, gradient communication, and synchronization.

This is why DataLoader and DDP lessons must stay connected but not conflated.


## DDP Training Path

A DataLoader candidate becomes training advice only after DDP validation.

The validation question is simple:

> Does the input-side candidate still matter when the model and distributed training loop are present?

The answer can differ by representation and platform.


## Interactive Moment 4: DDP Is The Training Truth

Use the dropdown to switch platform.

The one-node validation keeps the endpoint labels matched. The scale check then asks whether selected candidates still behave sensibly beyond one node.

The DDP story should stay short: one-node truth test, multi-node sanity check, and then back to the DataLoader decision framework.


In [ ]:
ddp_truth_fig().show()


## Synthetic GPU Input Is A Control

Synthetic GPU input removes the real input path.

It is useful because it shows a training-side ceiling: how fast the model loop can run when storage, decode, runtime preprocessing, and host-to-device input copy are removed.

Synthetic GPU input removes the real input path; it is not a dataset strategy.


In [ ]:
input_ceiling_fig().show()


## Decision Framework

1. Start with `canonical-224` PyTorch CPU for ordinary ImageNet-like input.
2. Tune enough to see the worker, batch, prefetch, and rank-balance shape.
3. Try DALI when large compressed images or heavy image preprocessing dominate.
4. Try prepared inputs only when repeated runs justify storage and precompute cost.
5. Use synthetic ceilings to separate input bottlenecks from model-side ceilings.
6. Validate the chosen path in DDP before recommending a training strategy.


## Support Staff Protocol

Ask these questions before prescribing an input path:

- What is actually on disk: original JPEG, pre-resized JPEG, prepared tensors, or something else?
- What transforms must remain online for scientific correctness?
- How many times will this dataset be reused?
- Is the user constrained by storage, CPU workers, GPU preprocessing, H2D, model compute, or synchronization?
- At what scale does the user actually train: one GPU, one node, or multi-node DDP?

Then prototype the smallest ladder that can isolate the bottleneck.


## What Not To Claim

Keep these guardrails in the main deck:

- Do not say DALI always wins.
- Do not say DataLoader-only wins are training wins.
- Do not recommend prepared fp16 as a general recipe.
- Do not claim the normal DALI JPEG path uses GDS.
- Do not compare full-resolution `1024` prepared or synthetic rows directly against JPEG rows that crop or resize to `224`.


## Reading Order After The Deck

Use the deck as a guided map, then read the public studies in this order:

1. DataLoader results summary.
2. Input pipeline reference and representation map.
3. Canonical single-GPU, one-node, and multi-node DataLoader progression.
4. DALI standard ImageNet and large JPEG crossover studies.
5. Prepared-input ceilings.
6. DDP DataLoader candidate follow-up and input ceilings.

The deck should teach intuition; the study pages remain the evidence record.
